<a href="https://colab.research.google.com/github/mnkshii/sign-language/blob/main/Copy_of_Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kaggle

In [2]:
import os
os.environ['KAGGLE_USERNAME'] = "heenadanu01"
os.environ['KAGGLE_KEY'] = "30630377"

In [3]:
!kaggle datasets download -d grassknoted/asl-alphabet

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
asl-alphabet.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
import os
print(os.path.exists("/content/asl_alphabet_train"))
# True  → dataset is there, skip download
# False → need to download again

True


In [5]:
!unzip -o asl-alphabet.zip

Streaming output truncated to the last 5000 lines.
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing19.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing190.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1900.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1901.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1902.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1903.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1904.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1905.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1906.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1907.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1908.jpg  
  inflating: asl_alphabet_train/asl_alphabet_train/nothing/nothing1909.jpg  
  inflating: asl_alphabet_tr

In [6]:
import os
print(os.listdir("/content"))

['.config', 'sign_language_lstm_final.keras', 'asl-alphabet.zip', 'asl_alphabet_train', 'wlasl-processed.zip', 'best_model.keras', 'asl_alphabet_test', 'sign_language_cnn_final.keras', 'best_lstm.keras', 'sample_data']


In [7]:

print(os.listdir("/content/asl_alphabet_train"))




['asl_alphabet_train']


In [8]:
TRAIN_DIR = "/content/asl_alphabet_train/asl_alphabet_train"
TEST_DIR  = "/content/asl_alphabet_test/asl_alphabet_test"

print(os.listdir(TRAIN_DIR))          # Should show A, B, C ... Z, space, etc.
print(os.listdir(TRAIN_DIR + "/A")[:5])  # Should show image filenames

['I', 'Z', 'P', 'T', 'M', 'H', 'F', 'O', 'C', 'B', 'E', 'L', 'K', 'Y', 'Q', 'D', 'G', 'W', 'J', 'space', 'N', 'del', 'R', 'X', 'U', 'A', 'V', 'S', 'nothing']
['A608.jpg', 'A2178.jpg', 'A1441.jpg', 'A745.jpg', 'A2608.jpg']


In [9]:
TRAIN_DIR = "/content/asl_alphabet_train"

In [10]:
import os

TRAIN_DIR = "/content/asl_alphabet_train"

classes = sorted(os.listdir(TRAIN_DIR))
print(f"Total classes : {len(classes)}")
print(f"Classes       : {classes}")


for cls in classes[:5]:  # preview first 5
    count = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    print(f"  {cls}: {count} images")

Total classes : 1
Classes       : ['asl_alphabet_train']
  asl_alphabet_train: 29 images


In [11]:

import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [12]:
import os, numpy as np, tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D,
                                      Flatten, Dense, Dropout,
                                      BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


tf.keras.backend.clear_session()

DATASET_PATH = "/content/asl_alphabet_train"
IMG_SIZE     = (64, 64)
BATCH_SIZE   = 64
EPOCHS       = 3        # just 3 epochs to test first

train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size = IMG_SIZE,
    color_mode  = 'grayscale',
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'training',
    seed        = 42
)

val_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size = IMG_SIZE,
    color_mode  = 'grayscale',
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'validation',
    seed        = 42
)

# Simple CNN
model = Sequential([
    Input(shape=(64, 64, 1)),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_gen.class_indices), activation='softmax')
])

model.compile(
    optimizer = 'adam',
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

steps_per_epoch  = max(1, len(train_gen) // 10)
validation_steps = max(1, len(val_gen)   // 10)

print(f"Steps per epoch  : {steps_per_epoch}")
print(f"Validation steps : {validation_steps}")
print(f"Classes          : {len(train_gen.class_indices)}")

history = model.fit(
    train_gen,
    validation_data  = val_gen,
    epochs           = EPOCHS,
    steps_per_epoch  = steps_per_epoch,
    validation_steps = validation_steps,
    callbacks        = [
        EarlyStopping(patience=2, restore_best_weights=True),
        ModelCheckpoint("best_model.keras", save_best_only=True)
    ]
)

model.save("sign_language_cnn_final.keras")
print("Done!")

Found 69600 images belonging to 1 classes.
Found 17400 images belonging to 1 classes.
Steps per epoch  : 108
Validation steps : 27
Classes          : 1
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


108/108 ━━━━━━━━━━━━━━━━━━━━ 14s 72ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/3
108/108 ━━━━━━━━━━━━━━━━━━━━ 8s 72ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/3
108/108 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Done!


In [13]:
import numpy as np
from tensorflow.keras.models import Model, load_model

# Reload model
model = load_model("best_model.keras")
print("Model loaded!")

# Fix — call model once with dummy input first
dummy = np.zeros((1, 64, 64, 1))
_ = model(dummy)
print("Model initialized!")

# Now print all layers
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name} → {layer.__class__.__name__}")

Model loaded!
Model initialized!
Layer 0: conv2d → Conv2D
Layer 1: batch_normalization → BatchNormalization
Layer 2: max_pooling2d → MaxPooling2D
Layer 3: conv2d_1 → Conv2D
Layer 4: batch_normalization_1 → BatchNormalization
Layer 5: max_pooling2d_1 → MaxPooling2D
Layer 6: flatten → Flatten
Layer 7: dense → Dense
Layer 8: dropout → Dropout
Layer 9: dense_1 → Dense


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


In [14]:
import os

# Find correct path
base = "/content/asl_alphabet_train"
contents = os.listdir(base)
print("Contents:", contents)

# If it shows another folder inside, go one level deeper
if len(contents) == 1 and os.path.isdir(os.path.join(base, contents[0])):
    DATASET_PATH = os.path.join(base, contents[0])
else:
    DATASET_PATH = base

print(f"Correct dataset path: {DATASET_PATH}")
print(f"Classes found: {os.listdir(DATASET_PATH)[:5]}")

Contents: ['asl_alphabet_train']
Correct dataset path: /content/asl_alphabet_train/asl_alphabet_train
Classes found: ['I', 'Z', 'P', 'T', 'M']


In [15]:

DATASET_PATH = "/content/asl_alphabet_train/asl_alphabet_train"

import os
classes = os.listdir(DATASET_PATH)
print(f"Total classes : {len(classes)}")
print(f"Classes       : {classes}")


Total classes : 29
Classes       : ['I', 'Z', 'P', 'T', 'M', 'H', 'F', 'O', 'C', 'B', 'E', 'L', 'K', 'Y', 'Q', 'D', 'G', 'W', 'J', 'space', 'N', 'del', 'R', 'X', 'U', 'A', 'V', 'S', 'nothing']


In [16]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

model = load_model("best_model.keras")
dummy = np.zeros((1, 64, 64, 1))
_ = model(dummy)
print("Model ready!")
# Feature extractor
inp = tf.keras.Input(shape=(64, 64, 1))
x   = inp
for layer in model.layers[:8]:
    x = layer(x)
feature_extractor = tf.keras.Model(inputs=inp, outputs=x)
feature_extractor.trainable = False
print(f"Feature extractor: {feature_extractor.output_shape}")

DATASET_PATH = "/content/asl_alphabet_train/asl_alphabet_train"
IMG_SIZE     = (64, 64)
BATCH_SIZE   = 64
SEQ_LEN      = 10
FEATURE_DIM  = 128

train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2
)
train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size = IMG_SIZE,
    color_mode  = 'grayscale',
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'training',
    seed        = 42
)
val_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size = IMG_SIZE,
    color_mode  = 'grayscale',
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    subset      = 'validation',
    seed        = 42
)
print(f"Classes: {len(train_gen.class_indices)}")  # should show 29

def extract_features(generator, steps):
    X_feat, y_labels = [], []
    for i in range(steps):
        X_batch, y_batch = generator[i]
        feats = feature_extractor(X_batch, training=False).numpy()
        X_feat.append(feats)
        y_labels.append(np.argmax(y_batch, axis=1))
    return np.concatenate(X_feat), np.concatenate(y_labels)

steps_train = max(1, len(train_gen) // 10)
steps_val   = max(1, len(val_gen)   // 10)

print("Extracting train features...")
X_flat_train, y_flat_train = extract_features(train_gen, steps_train)

print("Extracting val features...")
X_flat_val, y_flat_val = extract_features(val_gen, steps_val)

print(f"Train features: {X_flat_train.shape}")
print(f"Val   features: {X_flat_val.shape}")

def make_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(0, len(X) - seq_len, seq_len):
        X_seq.append(X[i : i + seq_len])
        y_seq.append(y[i + seq_len - 1])
    return np.array(X_seq, dtype="float32"), np.array(y_seq, dtype="int32")

X_train_seq, y_train_seq = make_sequences(X_flat_train, y_flat_train, SEQ_LEN)
X_val_seq,   y_val_seq   = make_sequences(X_flat_val,   y_flat_val,   SEQ_LEN)

print(f"\nLSTM Train : {X_train_seq.shape}")
print(f"LSTM Val   : {X_val_seq.shape}")


Model ready!
Feature extractor: (None, 128)
Found 69600 images belonging to 29 classes.
Found 17400 images belonging to 29 classes.
Classes: 29
Extracting train features...
Extracting val features...
Train features: (6912, 128)
Val   features: (1728, 128)

LSTM Train : (691, 10, 128)
LSTM Val   : (172, 10, 128)


In [17]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Bidirectional,
                                      Dense, Dropout,
                                      BatchNormalization,
                                      GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

NUM_CLASSES = len(train_gen.class_indices)  # 29
print(f"Number of classes: {NUM_CLASSES}")

# ── Build LSTM
inputs = Input(shape=(SEQ_LEN, FEATURE_DIM))

x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = GlobalAveragePooling1D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

lstm_model = Model(inputs, outputs)
lstm_model.compile(
    optimizer = tf.keras.optimizers.Adam(1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)
lstm_model.summary()

# ── Train
history_lstm = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data = (X_val_seq, y_val_seq),
    epochs          = 10,
    batch_size      = 32,
    callbacks       = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ModelCheckpoint("best_lstm.keras", save_best_only=True)
    ]
)

lstm_model.save("sign_language_lstm_final.keras")


Number of classes: 29


Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10, 256)        │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 10, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 10, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 449,309 (1.71 MB)

 Trainable params: 448,541 (1.71 MB)

 Non-trainable params: 768 (3.00 KB)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 52ms/step - accuracy: 0.0333 - loss: 3.5236 - val_accuracy: 0.0058 - val_loss: 3.3685
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.0666 - loss: 3.3060 - val_accuracy: 0.0640 - val_loss: 3.3716
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0854 - loss: 3.2343 - val_accuracy: 0.0349 - val_loss: 3.3712
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1114 - loss: 3.1342 - val_accuracy: 0.0291 - val_loss: 3.3784


In [18]:
import re
import numpy as np

IDX_TO_CLASS = {v: k for k, v in train_gen.class_indices.items()}
print(f"Classes mapped: {IDX_TO_CLASS}")

# ── NLP functions

def remove_duplicates(signs):
    """AAABBBCCC → ABC"""
    if not signs:
        return []
    result = [signs[0]]
    for s in signs[1:]:
        if s != result[-1]:
            result.append(s)
    return result

def signs_to_text(signs):

    text = ""
    for s in signs:
        if s.lower() == "space":
            text += " "
        elif s.lower() in ("del", "delete"):
            text = text[:-1]
        elif s.lower() == "nothing":
            continue
        else:
            text += s.upper()
    return re.sub(r'\s+', ' ', text).strip()

def correct_grammar(text):
    """
    Rule based correction:
    - Capitalize first letter
    - Add period at end
    - Fix common words
    """
    if not text:
        return text

    # Common ASL word corrections
    corrections = {
        "U"    : "you",
        "R"    : "are",
        "UR"   : "you are",
        "THX"  : "thanks",
        "PLS"  : "please",
        "HRU"  : "how are you",
        "GM"   : "good morning",
        "GN"   : "good night",
        "HI"   : "hi",
        "BYE"  : "bye",
        "YES"  : "yes",
        "NO"   : "no",
        "OK"   : "okay",
        "TY"   : "thank you",
        "NP"   : "no problem",
        "IMO"  : "in my opinion",
        "ILY"  : "I love you",
        "NAME" : "name",
        "HELP" : "help",
        "STOP" : "stop",
        "GO"   : "go",
        "COME" : "come",
        "WANT" : "want",
        "NEED" : "need",
        "LIKE" : "like",
        "GOOD" : "good",
        "BAD"  : "bad",
    }

    # Check if full text matches a known word
    upper = text.upper().strip()
    if upper in corrections:
        text = corrections[upper]

    # Capitalize first letter
    text = text.strip().capitalize()

    # Add period at end if missing
    if text and not text[-1] in ('.', '!', '?'):
        text += '.'

    return text

def nlp_pipeline(predicted_indices):
    """Full pipeline: indices → corrected sentence"""
    # Step 1 — indices to sign labels
    signs = [IDX_TO_CLASS[i] for i in predicted_indices]
    print(f"  Raw signs  : {signs}")

    # Step 2 — remove duplicates
    signs = remove_duplicates(signs)
    print(f"  Deduped    : {signs}")

    # Step 3 — signs to text
    raw = signs_to_text(signs)
    print(f"  Raw text   : {raw}")

    # Step 4 — correct grammar
    corrected = correct_grammar(raw)
    print(f"  Corrected  : {corrected}")

    return corrected

# ── Test on 5 validation samples ───────────────────────────
print("\n--- Testing NLP on 5 samples ---\n")
for i in range(5):
    seq   = X_val_seq[i:i+1]
    probs = lstm_model.predict(seq, verbose=0)
    pred  = int(np.argmax(probs[0]))
    conf  = float(probs[0][pred])
    print(f"Sample {i+1} (confidence: {conf:.2f}):")
    result = nlp_pipeline([pred])
    print(f"  Final      : '{result}'")
    print()

print("Cell 8 Done! ✅")

Classes mapped: {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I', 9: 'J', 10: 'K', 11: 'L', 12: 'M', 13: 'N', 14: 'O', 15: 'P', 16: 'Q', 17: 'R', 18: 'S', 19: 'T', 20: 'U', 21: 'V', 22: 'W', 23: 'X', 24: 'Y', 25: 'Z', 26: 'del', 27: 'nothing', 28: 'space'}

--- Testing NLP on 5 samples ---

Sample 1 (confidence: 0.04):
  Raw signs  : ['X']
  Deduped    : ['X']
  Raw text   : X
  Corrected  : X.
  Final      : 'X.'

Sample 2 (confidence: 0.04):
  Raw signs  : ['X']
  Deduped    : ['X']
  Raw text   : X
  Corrected  : X.
  Final      : 'X.'

Sample 3 (confidence: 0.04):
  Raw signs  : ['X']
  Deduped    : ['X']
  Raw text   : X
  Corrected  : X.
  Final      : 'X.'

Sample 4 (confidence: 0.04):
  Raw signs  : ['X']
  Deduped    : ['X']
  Raw text   : X
  Corrected  : X.
  Final      : 'X.'

Sample 5 (confidence: 0.04):
  Raw signs  : ['X']
  Deduped    : ['X']
  Raw text   : X
  Corrected  : X.
  Final      : 'X.'

Cell 8 Done! ✅


In [19]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [20]:
!pip install mediapipe

In [21]:
import mediapipe as mp
print(mp.__version__)


0.10.33


In [22]:
from google.colab.patches import cv2_imshow
from google.colab import output
from IPython.display import display, Javascript
import base64

# Use JavaScript to capture from browser webcam
def capture_webcam_colab():
    js = Javascript('''
        async function capture() {
            const video = document.createElement("video");
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            const canvas = document.createElement("canvas");
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext("2d").drawImage(video, 0, 0);

            stream.getTracks().forEach(t => t.stop());
            return canvas.toDataURL("image/jpeg", 0.8);
        }
        capture().then(data => google.colab.kernel.invokeFunction("notebook.get_frame", [data], {}));
    ''')

    frame_data = {}
    def save_frame(data):
        frame_data["img"] = data
        output.eval_js("0")  # flush

    output.register_callback("notebook.get_frame", save_frame)
    display(js)
    return frame_data

# ── Single frame prediction (Colab-friendly) ───────────────
import cv2
import numpy as np
from collections import deque

sequence = deque(maxlen=30)
sentence = []

def predict_from_frame(frame_b64):
    # Decode base64 image
    img_data = base64.b64decode(frame_b64.split(",")[1])
    np_arr   = np.frombuffer(img_data, np.uint8)
    frame    = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_hands = mp.solutions.hands
    with mp_hands.Hands(max_num_hands=1,
                        min_detection_confidence=0.7) as hands:
        result    = hands.process(rgb)
        keypoints = np.zeros(63)

        if result.multi_hand_landmarks:
            lm = result.multi_hand_landmarks[0]
            keypoints = np.array(
                [[p.x, p.y, p.z] for p in lm.landmark]
            ).flatten()

    sequence.append(keypoints)

    if len(sequence) == 30:
        seq_input = np.expand_dims(list(sequence), axis=0)
        probs     = lstm_model.predict(seq_input, verbose=0)
        pred      = int(np.argmax(probs[0]))
        conf      = float(probs[0][pred])

        if conf >= 0.75:
            sentence.append(pred)
            result_text = nlp_pipeline(sentence)
            print(f"Confidence: {conf:.2f} → {result_text}")
    else:
        print(f"Buffering... {len(sequence)}/30 frames")

# ── Run: call this repeatedly to accumulate frames ─────────
capture_webcam_colab()
print("Cell 9 Done! ✅")

<IPython.core.display.Javascript object>

Cell 9 Done! ✅


In [25]:
# Step 1 — Install dependencies
!pip install flask flask-ngrok pyngrok -q

In [26]:
# Step 1 — Install
!npm install -g localtunnel
!pip install flask -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [ ]:
from IPython.display import display, HTML

html_code = """

<!DOCTYPE html>
<html>
<head>
    <title>ASL Live Detection</title>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/control_utils/control_utils.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js"></script>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            background: #1a1a2e;
            font-family: 'Segoe UI', sans-serif;
            color: white;
            display: flex;
            flex-direction: column;
            align-items: center;
            padding: 20px;
            min-height: 100vh;
        }
        h1 { color: #4CAF50; margin-bottom: 15px; font-size: 28px; }

        #layout {
            display: flex;
            gap: 20px;
            align-items: flex-start;
            flex-wrap: wrap;
            justify-content: center;
        }

        #output_canvas {
            border-radius: 14px;
            border: 3px solid #4CAF50;
            box-shadow: 0 0 30px #4CAF5055;
        }

        #right_panel {
            display: flex;
            flex-direction: column;
            gap: 15px;
            width: 260px;
        }

        #sign_box {
            background: #0f3460;
            border-radius: 14px;
            padding: 20px;
            text-align: center;
            border: 2px solid #4CAF50;
        }

        #sign_display {
            font-size: 100px;
            font-weight: bold;
            color: #4CAF50;
            text-shadow: 0 0 25px #4CAF50;
            line-height: 1.1;
        }

        #sign_label {
            font-size: 13px;
            color: #aaa;
            margin-top: 5px;
        }

        #conf_wrap {
            background: #333;
            border-radius: 10px;
            height: 18px;
            overflow: hidden;
            margin-top: 10px;
        }

        #conf_fill {
            background: linear-gradient(90deg, #4CAF50, #8BC34A);
            height: 100%;
            width: 0%;
            transition: width 0.2s ease;
            border-radius: 10px;
        }

        #hold_wrap {
            background: #333;
            border-radius: 10px;
            height: 10px;
            overflow: hidden;
            margin-top: 6px;
        }

        #hold_fill {
            background: #f39c12;
            height: 100%;
            width: 0%;
            transition: width 0.1s linear;
            border-radius: 10px;
        }

        #hold_label { font-size: 11px; color: #f39c12; margin-top: 3px; }

        #history_box {
            background: #0f3460;
            border-radius: 14px;
            padding: 15px;
            border: 2px solid #3498db;
        }

        #history_title { font-size: 13px; color: #3498db; margin-bottom: 8px; }

        #history_list {
            display: flex;
            flex-wrap: wrap;
            gap: 5px;
        }

        .history_chip {
            background: #1a4a7a;
            border-radius: 6px;
            padding: 3px 8px;
            font-size: 16px;
            font-weight: bold;
        }

        #status {
            font-size: 13px;
            color: #aaa;
            margin-top: 5px;
            text-align: center;
        }

        #text_section {
            width: 100%;
            max-width: 780px;
            margin-top: 15px;
        }

        #text_box {
            background: #0f3460;
            border-radius: 12px;
            padding: 18px 22px;
            font-size: 30px;
            letter-spacing: 4px;
            min-height: 64px;
            word-break: break-all;
            border: 2px solid #4CAF50;
            width: 100%;
        }

        #cursor {
            display: inline-block;
            width: 3px;
            height: 34px;
            background: #4CAF50;
            margin-left: 4px;
            vertical-align: middle;
            animation: blink 1s infinite;
        }

        @keyframes blink { 0%,100%{opacity:1} 50%{opacity:0} }

        .btn-row {
            display: flex;
            gap: 10px;
            margin-top: 10px;
            justify-content: center;
        }

        .btn {
            padding: 10px 22px;
            font-size: 15px;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-weight: bold;
            transition: transform 0.1s, opacity 0.2s;
        }
        .btn:hover { transform: scale(1.05); opacity: 0.9; }
        .btn:active { transform: scale(0.97); }
    </style>
</head>
<body>
    <h1>🤚 Live ASL Detection</h1>

    <div id="layout">
        <div>
            <video id="input_video" style="display:none;"></video>
            <canvas id="output_canvas" width="500" height="375"></canvas>
            <div id="status">⏳ Starting camera...</div>
        </div>

        <div id="right_panel">
            <div id="sign_box">
                <div id="sign_display">–</div>
                <div id="sign_label">Current Sign</div>
                <div id="conf_wrap"><div id="conf_fill"></div></div>
                <div style="font-size:11px;color:#aaa;margin-top:3px;">Confidence</div>
                <div id="hold_wrap"><div id="hold_fill"></div></div>
                <div id="hold_label">Hold to type...</div>
            </div>

            <div id="history_box">
                <div id="history_title">📋 Last 10 Signs</div>
                <div id="history_list"></div>
            </div>
        </div>
    </div>

    <div id="text_section">
        <div id="text_box"><span id="text_content"></span><span id="cursor"></span></div>
        <div class="btn-row">
            <button class="btn" style="background:#e74c3c;color:white" onclick="clearText()">🗑 Clear</button>
            <button class="btn" style="background:#3498db;color:white" onclick="addSpace()">␣ Space</button>
            <button class="btn" style="background:#f39c12;color:white" onclick="deleteLast()">⌫ Delete</button>
            <button class="btn" style="background:#9b59b6;color:white" onclick="copyText()">📋 Copy</button>
        </div>
    </div>

    <script>
        const video     = document.getElementById('input_video');
        const canvas    = document.getElementById('output_canvas');
        const ctx       = canvas.getContext('2d');
        const signDisp  = document.getElementById('sign_display');
        const textContent = document.getElementById('text_content');
        const confFill  = document.getElementById('conf_fill');
        const holdFill  = document.getElementById('hold_fill');
        const holdLabel = document.getElementById('hold_label');
        const statusEl  = document.getElementById('status');
        const histList  = document.getElementById('history_list');

        let currentText    = "";
        let lastSign       = "";
        let signHoldFrames = 0;
        let lastAdded      = "";
        let signHistory    = [];
        const HOLD_THRESHOLD = 25;

        function updateText() { textContent.innerText = currentText; }
        function clearText()  { currentText = ""; updateText(); }
        function addSpace()   { currentText += " "; updateText(); }
        function deleteLast() { currentText = currentText.slice(0,-1); updateText(); }
        function copyText()   {
            navigator.clipboard.writeText(currentText);
            statusEl.innerText = "✅ Copied to clipboard!";
            setTimeout(() => statusEl.innerText = "", 2000);
        }

        function addToHistory(sign) {
            signHistory.push(sign);
            if (signHistory.length > 10) signHistory.shift();
            histList.innerHTML = signHistory
                .map(s => `<div class="history_chip">${s}</div>`)
                .join('');
        }

        function dist(a, b) {
            return Math.sqrt((a.x-b.x)**2 + (a.y-b.y)**2 + (a.z-b.z)**2);
        }
        function fingerUp(tip, pip) { return tip.y < pip.y; }

        function classifyASL(L) {
            const thumbTip=L[4], thumbIP=L[3], thumbMCP=L[2], thumbCMC=L[1];
            const idxTip=L[8],  idxPIP=L[6],  idxMCP=L[5];
            const midTip=L[12], midPIP=L[10], midMCP=L[9];
            const rngTip=L[16], rngPIP=L[14];
            const pnkTip=L[20], pnkPIP=L[18];
            const wrist=L[0];

            const iUp = fingerUp(idxTip, idxPIP);
            const mUp = fingerUp(midTip, midPIP);
            const rUp = fingerUp(rngTip, rngPIP);
            const pUp = fingerUp(pnkTip, pnkPIP);

            const thumbUp   = thumbTip.y < thumbIP.y;
            const thumbOut  = dist(thumbTip, idxTip) > 0.15;
            const allCurled = !iUp && !mUp && !rUp && !pUp;
            const allUp     =  iUp &&  mUp &&  rUp &&  pUp;

            const pinchDist = dist(thumbTip, idxTip);

            if (allCurled && thumbOut && !thumbUp)       return {sign:"A", conf:90};
            if (allUp && !thumbOut)                      return {sign:"B", conf:87};
            if (!allCurled && !allUp && !iUp && !pUp) {
                if (dist(thumbTip,idxTip)<0.22 && dist(thumbTip,pnkTip)<0.30)
                    return {sign:"C", conf:83};
            }
            if (iUp && !mUp && !rUp && !pUp && dist(thumbTip,midTip)<0.07)
                return {sign:"D", conf:82};
            if (allCurled && !thumbOut && pinchDist < 0.08)
                return {sign:"E", conf:80};
            if (!iUp && mUp && rUp && pUp && pinchDist < 0.06)
                return {sign:"F", conf:82};
            if (iUp && !mUp && !rUp && !pUp && thumbOut &&
                Math.abs(idxTip.x - idxMCP.x) > Math.abs(idxTip.y - idxMCP.y))
                return {sign:"G", conf:75};
            if (iUp && mUp && !rUp && !pUp &&
                Math.abs(idxTip.x - idxMCP.x) > Math.abs(idxTip.y - idxMCP.y))
                return {sign:"H", conf:75};
            if (!iUp && !mUp && !rUp && pUp && !thumbOut)
                return {sign:"I", conf:84};
            if (iUp && mUp && !rUp && !pUp && thumbOut && pinchDist > 0.1)
                return {sign:"K", conf:74};
            if (iUp && !mUp && !rUp && !pUp && thumbOut)
                return {sign:"L", conf:90};
            if (allCurled && !thumbOut && dist(thumbTip, wrist) < 0.15)
                return {sign:"M", conf:72};
            if (!allUp && !allCurled && pinchDist < 0.07 &&
                dist(midTip, thumbTip) < 0.09)
                return {sign:"O", conf:80};
            if (iUp && !mUp && !rUp && !pUp && thumbOut &&
                idxTip.y > idxMCP.y)
                return {sign:"P", conf:73};
            if (iUp && mUp && !rUp && !pUp && !thumbOut &&
                Math.abs(idxTip.x - midTip.x) < 0.04)
                return {sign:"R", conf:76};
            if (allCurled && !thumbOut && thumbTip.y < idxPIP.y)
                return {sign:"S", conf:80};
            if (iUp && mUp && !rUp && !pUp && !thumbOut &&
                dist(idxTip, midTip) < 0.06)
                return {sign:"U", conf:80};
            if (iUp && mUp && !rUp && !pUp && !thumbOut &&
                dist(idxTip, midTip) > 0.07)
                return {sign:"V", conf:86};
            if (iUp && mUp && rUp && !pUp)
                return {sign:"W", conf:84};
            if (!iUp && !mUp && !rUp && !pUp && !thumbOut &&
                dist(idxTip, idxPIP) < 0.06)
                return {sign:"X", conf:72};
            if (!iUp && !mUp && !rUp && pUp && thumbOut)
                return {sign:"Y", conf:86};
            if (allUp && thumbOut)
                return {sign:"5", conf:82};

            return {sign:"?", conf:0};
        }

        const hands = new Hands({
            locateFile: f => `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${f}`
        });

        hands.setOptions({
            maxNumHands: 1,
            modelComplexity: 1,
            minDetectionConfidence: 0.65,
            minTrackingConfidence: 0.55
        });

        hands.onResults(results => {
            ctx.save();
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            ctx.translate(canvas.width, 0);
            ctx.scale(-1, 1);
            ctx.drawImage(results.image, 0, 0, canvas.width, canvas.height);
            ctx.restore();

            if (results.multiHandLandmarks?.length > 0) {
                const lm = results.multiHandLandmarks[0];

                ctx.save();
                ctx.translate(canvas.width, 0);
                ctx.scale(-1, 1);
                drawConnectors(ctx, lm, HAND_CONNECTIONS, {color:'#00FF88', lineWidth:3});
                drawLandmarks(ctx, lm, {color:'#FF4444', lineWidth:1, radius:5});
                ctx.restore();

                const {sign, conf} = classifyASL(lm);

                signDisp.innerText       = sign;
                confFill.style.width     = conf + '%';

                if (sign === lastSign && sign !== "?") {
                    signHoldFrames++;
                    const pct = Math.min(100, (signHoldFrames / HOLD_THRESHOLD) * 100);
                    holdFill.style.width = pct + '%';
                    holdLabel.innerText  = signHoldFrames >= HOLD_THRESHOLD
                        ? `✅ "${sign}" added!` : `Hold... ${Math.round(pct)}%`;

                    if (signHoldFrames === HOLD_THRESHOLD && sign !== lastAdded) {
                        currentText += sign;
                        updateText();
                        addToHistory(sign);
                        lastAdded = sign;
                    }
                } else {
                    lastSign       = sign;
                    signHoldFrames = 0;
                    holdFill.style.width = '0%';
                    holdLabel.innerText  = 'Hold to type...';
                    if (sign !== lastAdded) lastAdded = "";
                }

                statusEl.innerText = sign === "?"
                    ? "🤔 Sign not recognized"
                    : `Confidence: ${conf}%`;

            } else {
                signDisp.innerText       = "–";
                confFill.style.width     = "0%";
                holdFill.style.width     = "0%";
                holdLabel.innerText      = "Hold to type...";
                statusEl.innerText       = "👋 Show your hand to the camera";
                signHoldFrames = 0;
            }
        });

        const camera = new Camera(video, {
            onFrame: async () => { await hands.send({image: video}); },
            width: 500, height: 375
        });

        camera.start()
            .then(() => { statusEl.innerText = "✅ Camera running — show a hand sign!"; })
            .catch(e  => { statusEl.innerText = "❌ Camera error: " + e.message; });
    </script>
</body>
</html>
"""

display(HTML(html_code))

In [35]:
from flask import Flask
import threading, subprocess, time

html = '''<!DOCTYPE html>
<html>
<head>
    <title>ASL Live Detection</title>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/control_utils/control_utils.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js"></script>
    <style>
        *{box-sizing:border-box;margin:0;padding:0}
        body{background:#1a1a2e;font-family:"Segoe UI",sans-serif;color:white;display:flex;flex-direction:column;align-items:center;padding:20px;min-height:100vh}
        h1{color:#4CAF50;margin-bottom:15px;font-size:28px}
        #layout{display:flex;gap:20px;align-items:flex-start;flex-wrap:wrap;justify-content:center}
        #output_canvas{border-radius:14px;border:3px solid #4CAF50;box-shadow:0 0 30px #4CAF5055}
        #right_panel{display:flex;flex-direction:column;gap:15px;width:260px}
        #sign_box{background:#0f3460;border-radius:14px;padding:20px;text-align:center;border:2px solid #4CAF50}
        #sign_display{font-size:100px;font-weight:bold;color:#4CAF50;text-shadow:0 0 25px #4CAF50;line-height:1.1}
        #conf_wrap{background:#333;border-radius:10px;height:18px;overflow:hidden;margin-top:10px}
        #conf_fill{background:linear-gradient(90deg,#4CAF50,#8BC34A);height:100%;width:0%;transition:width 0.2s;border-radius:10px}
        #hold_wrap{background:#333;border-radius:10px;height:10px;overflow:hidden;margin-top:6px}
        #hold_fill{background:#f39c12;height:100%;width:0%;transition:width 0.1s;border-radius:10px}
        #hold_label{font-size:11px;color:#f39c12;margin-top:3px}
        #history_box{background:#0f3460;border-radius:14px;padding:15px;border:2px solid #3498db}
        .history_chip{background:#1a4a7a;border-radius:6px;padding:3px 8px;font-size:16px;font-weight:bold;display:inline-block;margin:2px}
        #status{font-size:13px;color:#aaa;margin-top:5px;text-align:center}
        #text_section{width:100%;max-width:780px;margin-top:15px}
        #text_box{background:#0f3460;border-radius:12px;padding:18px 22px;font-size:30px;letter-spacing:4px;min-height:64px;word-break:break-all;border:2px solid #4CAF50;width:100%}
        #cursor{display:inline-block;width:3px;height:34px;background:#4CAF50;margin-left:4px;vertical-align:middle;animation:blink 1s infinite}
        @keyframes blink{0%,100%{opacity:1}50%{opacity:0}}
        .btn-row{display:flex;gap:10px;margin-top:10px;justify-content:center}
        .btn{padding:10px 22px;font-size:15px;border:none;border-radius:8px;cursor:pointer;font-weight:bold;transition:transform 0.1s}
        .btn:hover{transform:scale(1.05)}
    </style>
</head>
<body>
    <h1>🤚 Live ASL Detection</h1>
    <div id="layout">
        <div style="display:flex;flex-direction:column;align-items:center;">
            <video id="input_video" style="display:none;"></video>
            <canvas id="output_canvas" width="500" height="375"></canvas>
            <div id="status">⏳ Starting camera...</div>
        </div>
        <div id="right_panel">
            <div id="sign_box">
                <div id="sign_display">–</div>
                <div style="font-size:13px;color:#aaa">Current Sign</div>
                <div id="conf_wrap"><div id="conf_fill"></div></div>
                <div style="font-size:11px;color:#aaa;margin-top:3px">Confidence</div>
                <div id="hold_wrap"><div id="hold_fill"></div></div>
                <div id="hold_label">Hold to type...</div>
            </div>
            <div id="history_box">
                <div style="font-size:13px;color:#3498db;margin-bottom:8px">📋 Last 10 Signs</div>
                <div id="history_list"></div>
            </div>
        </div>
    </div>
    <div id="text_section">
        <div id="text_box"><span id="text_content"></span><span id="cursor"></span></div>
        <div class="btn-row">
            <button class="btn" style="background:#e74c3c;color:white" onclick="clearText()">🗑 Clear</button>
            <button class="btn" style="background:#3498db;color:white" onclick="addSpace()">␣ Space</button>
            <button class="btn" style="background:#f39c12;color:white" onclick="deleteLast()">⌫ Delete</button>
            <button class="btn" style="background:#9b59b6;color:white" onclick="copyText()">📋 Copy</button>
        </div>
    </div>

<script>
const video      = document.getElementById("input_video");
const canvas     = document.getElementById("output_canvas");
const ctx        = canvas.getContext("2d");
const signDisp   = document.getElementById("sign_display");
const textContent= document.getElementById("text_content");
const confFill   = document.getElementById("conf_fill");
const holdFill   = document.getElementById("hold_fill");
const holdLabel  = document.getElementById("hold_label");
const statusEl   = document.getElementById("status");
const histList   = document.getElementById("history_list");

// Use a separate offscreen canvas for mediapipe
// so the mirror transform NEVER touches the main canvas state
const offscreen  = document.createElement("canvas");
offscreen.width  = 500;
offscreen.height = 375;
const offCtx     = offscreen.getContext("2d");

let currentText="", lastSign="", signHoldFrames=0, lastAdded="", signHistory=[];
const HOLD_THRESHOLD = 10;

function updateText(){ textContent.innerText = currentText; }
function clearText() { currentText=""; updateText(); }
function addSpace()  { currentText+=" "; updateText(); }
function deleteLast(){ currentText=currentText.slice(0,-1); updateText(); }
function copyText()  {
    navigator.clipboard.writeText(currentText);
    statusEl.innerText="✅ Copied!";
    setTimeout(()=>statusEl.innerText="",2000);
}
function addToHistory(s){
    signHistory.push(s);
    if(signHistory.length>10) signHistory.shift();
    histList.innerHTML=signHistory.map(x=>`<div class="history_chip">${x}</div>`).join("");
}

function dist(a,b){ return Math.sqrt((a.x-b.x)**2+(a.y-b.y)**2+(a.z-b.z)**2); }
function fingerUp(tip,pip){ return tip.y < pip.y; }

function classifyASL(L){
    try{
        const tTip=L[4],iTip=L[8],iPIP=L[6],mTip=L[12],mPIP=L[10],
              rTip=L[16],rPIP=L[14],pTip=L[20],pPIP=L[18];
        const iUp=fingerUp(iTip,iPIP), mUp=fingerUp(mTip,mPIP),
              rUp=fingerUp(rTip,rPIP), pUp=fingerUp(pTip,pPIP);
        const thumbOut=dist(tTip,iTip)>0.15,
              allCurled=!iUp&&!mUp&&!rUp&&!pUp,
              allUp=iUp&&mUp&&rUp&&pUp,
              pinch=dist(tTip,iTip);

        if(allCurled&&thumbOut&&tTip.y>L[3].y)           return{sign:"A",conf:90};
        if(allUp&&!thumbOut)                              return{sign:"B",conf:87};
        if(!allCurled&&!allUp&&!iUp&&!pUp
           &&dist(tTip,iTip)<0.22&&dist(tTip,pTip)<0.30) return{sign:"C",conf:83};
        if(iUp&&!mUp&&!rUp&&!pUp&&dist(tTip,mTip)<0.07)  return{sign:"D",conf:82};
        if(allCurled&&!thumbOut&&pinch<0.08)              return{sign:"E",conf:80};
        if(!iUp&&mUp&&rUp&&pUp&&pinch<0.06)              return{sign:"F",conf:82};
        if(!iUp&&!mUp&&!rUp&&pUp&&!thumbOut)             return{sign:"I",conf:84};
        if(iUp&&!mUp&&!rUp&&!pUp&&thumbOut)              return{sign:"L",conf:90};
        if(!allUp&&!allCurled&&pinch<0.07
           &&dist(mTip,tTip)<0.09)                        return{sign:"O",conf:80};
        if(iUp&&mUp&&!rUp&&!pUp&&!thumbOut
           &&dist(iTip,mTip)<0.04)                        return{sign:"R",conf:76};
        if(allCurled&&!thumbOut&&tTip.y<iPIP.y)          return{sign:"S",conf:80};
        if(iUp&&mUp&&!rUp&&!pUp&&!thumbOut
           &&dist(iTip,mTip)<0.06)                        return{sign:"U",conf:80};
        if(iUp&&mUp&&!rUp&&!pUp&&!thumbOut
           &&dist(iTip,mTip)>0.07)                        return{sign:"V",conf:86};
        if(iUp&&mUp&&rUp&&!pUp)                          return{sign:"W",conf:84};
        if(!iUp&&!mUp&&!rUp&&pUp&&thumbOut)              return{sign:"Y",conf:86};
        if(allUp&&thumbOut)                               return{sign:"5",conf:82};
    }catch(e){}
    return{sign:"?",conf:0};
}

function onResults(results){
    try{
        // ── 1. Draw mirrored video onto OFFSCREEN canvas ──
        offCtx.save();
        offCtx.clearRect(0,0,500,375);
        offCtx.translate(500,0);
        offCtx.scale(-1,1);
        offCtx.drawImage(results.image,0,0,500,375);
        offCtx.restore();                          // ← transform fully reset here

        // ── 2. Draw landmarks onto OFFSCREEN canvas ───────
        if(results.multiHandLandmarks?.length > 0){
            const lm = results.multiHandLandmarks[0];
            offCtx.save();
            offCtx.translate(500,0);
            offCtx.scale(-1,1);
            drawConnectors(offCtx,lm,HAND_CONNECTIONS,{color:"#00FF88",lineWidth:3});
            drawLandmarks(offCtx,lm,{color:"#FF4444",lineWidth:1,radius:5});
            offCtx.restore();                      // ← transform fully reset here
        }

        // ── 3. Copy offscreen → main canvas (NO transforms) ──
        ctx.clearRect(0,0,canvas.width,canvas.height);
        ctx.drawImage(offscreen,0,0);              // ← plain copy, no transforms

        // ── 4. Classify ───────────────────────────────────
        if(results.multiHandLandmarks?.length > 0){
            const lm = results.multiHandLandmarks[0];
            const {sign,conf} = classifyASL(lm);
            signDisp.innerText   = sign;
            confFill.style.width = conf + "%";

            if(sign === lastSign && sign !== "?"){
                signHoldFrames++;
                const pct = Math.min(100,(signHoldFrames/HOLD_THRESHOLD)*100);
                holdFill.style.width = pct + "%";
                holdLabel.innerText  = signHoldFrames >= HOLD_THRESHOLD
                    ? `✅ "${sign}" added!`
                    : `Hold... ${Math.round(pct)}%`;
                if(signHoldFrames === HOLD_THRESHOLD && sign !== lastAdded){
                    currentText += sign;
                    updateText();
                    addToHistory(sign);
                    lastAdded = sign;
                }
            } else {
                lastSign       = sign;
                signHoldFrames = 0;
                holdFill.style.width = "0%";
                holdLabel.innerText  = "Hold to type...";
                if(sign !== lastAdded) lastAdded = "";
            }
            statusEl.innerText = sign==="?" ? "🤔 Not recognized" : `Confidence: ${conf}%`;

        } else {
            signDisp.innerText   = "–";
            confFill.style.width = "0%";
            holdFill.style.width = "0%";
            holdLabel.innerText  = "Hold to type...";
            statusEl.innerText   = "👋 Show your hand to the camera";
            signHoldFrames = 0;
        }
    }catch(err){
        console.warn("Frame skipped:",err);
    }
}

// ── MediaPipe + Camera ─────────────────────────────────────
const hands = new Hands({
    locateFile: f => `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${f}`
});
hands.setOptions({
    maxNumHands:1,
    modelComplexity:1,
    minDetectionConfidence:0.6,
    minTrackingConfidence:0.5
});
hands.onResults(onResults);

const camera = new Camera(video, {
    onFrame: async () => {
        try{ await hands.send({image: video}); }
        catch(e){ console.warn("Send skipped:",e); }
    },
    width:500, height:375
});

camera.start()
    .then(()=> statusEl.innerText="✅ Camera running — show a hand sign!")
    .catch(e => statusEl.innerText="❌ "+e.message);
</script>
</body>
</html>'''

with open('/content/asl_live.html', 'w') as f:
    f.write(html)

app = Flask(__name__)

@app.route('/')
def index():
    with open('/content/asl_live.html') as f:
        return f.read()

threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False),
    daemon=True
).start()

time.sleep(2)

proc = subprocess.Popen(['lt','--port','5000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)
line = proc.stdout.readline().decode().strip()
print(f"\n✅ Open this link:\n\n   👉  {line}  👈\n")
print("Allow camera → show hand sign → hold steady to type!")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



✅ Open this link:

   👉  your url is: https://fluffy-apples-sink.loca.lt  👈

Allow camera → show hand sign → hold steady to type!


In [36]:
# Run this to get your IP if localtunnel asks for a password
!curl ifconfig.me

34.124.217.238